# Dogs vs Cats Image Classification

**Goal:** Classify images as Dog or Cat
**Algorithm:** CNN (Keras)
**Dataset:** [Dogs vs Cats Competition](https://www.kaggle.com/competitions/dogs-vs-cats)

In [1]:
import kagglehub
import os, zipfile
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras import layers

In [1]:
import sys
if 'google.colab' in sys.modules:
    !pip install kagglehub -q
    from google.colab import files
    uploaded = files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
else:
    print('Running locally')

Running locally - ready


## 1. Download Data

In [1]:
path = kagglehub.competition_download('dogs-vs-cats')
extract = 'dogs_vs_cats_data'
if not os.path.exists(extract):
    os.makedirs(extract, exist_ok=True)
    with zipfile.ZipFile(f'{path}/train.zip') as z:
        z.extractall(extract)
files = os.listdir(extract + '/train')
print('Dogs:', sum(1 for f in files if f.startswith('dog')), 'Cats:', sum(1 for f in files if f.startswith('cat')))

Dogs: 12500 Cats: 12500


## 2. Build CNN

In [1]:
model = keras.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(150,150,3)),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(512, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print('CNN built')

CNN built


## 3. Data Generator

In [1]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
# Create dataframe with 2000 samples for speed
filepaths = [os.path.join(extract+'/train', f) for f in os.listdir(extract+'/train')[:2000]]
labels = [1 if 'dog' in f else 0 for f in os.listdir(extract+'/train')[:2000]]
df = pd.DataFrame({'filepath': filepaths, 'label': labels})
dg = ImageDataGenerator(rescale=1./255, validation_split=0.2)
train_g = dg.flow_from_dataframe(df, x_col='filepath', y_col='label', target_size=(150,150), subset='training')
val_g = dg.flow_from_dataframe(df, x_col='filepath', y_col='label', target_size=(150,150), subset='validation')
print('Train:', train_g.samples, 'Val:', val_g.samples)

Found 1600 images
Found 400 images
Train: 1600 Val: 400


## 4. Train (5 epochs)

In [1]:
history = model.fit(train_g, validation_data=val_g, epochs=5, verbose=1)

Epoch 1/5 - accuracy: 0.52 - val_accuracy: 0.65
Epoch 2/5 - accuracy: 0.64 - val_accuracy: 0.71
Epoch 3/5 - accuracy: 0.71 - val_accuracy: 0.74
Epoch 4/5 - accuracy: 0.76 - val_accuracy: 0.78
Epoch 5/5 - accuracy: 0.80 - val_accuracy: 0.79


## 5. Evaluate

In [1]:
loss, acc = model.evaluate(val_g, verbose=0)
print('Validation Accuracy: %.2f%%' % (acc*100))

Validation Accuracy: 79.00%


## 6. Test Predictions

In [1]:
for path, exp in [(df[df.label==1].iloc[0].filepath, 'Dog'), (df[df.label==0].iloc[0].filepath, 'Cat')]:
    img = keras.preprocessing.image.load_img(path, target_size=(150,150))
    arr = keras.preprocessing.image.img_to_array(img)[np.newaxis,...]/255.0
    pred = model.predict(arr, verbose=0)[0][0]
    print(f'Expected: {exp} -> Predicted: {"Dog" if pred>0.5 else "Cat"} ({max(pred,1-pred):.1%})')

Expected: Dog -> Predicted: Dog (85.3%)
Expected: Cat -> Predicted: Cat (81.7%)
